# Exploratory Data Analysis Report. Group 6a.
---

### Main Dataset Overview

The main dataset called `supermarket.csv` contains **100,000 customer shopping trips**, of which
approximately **15% are frauds**. Each record encodes the full
sequence of a customer's in-store journey as a list of scan events, where
each event captures the **department visited**, **time elapsed**, and
**amount registered**.

---

### Datasets

| File | Description | Fraud status |
|---|---|---|
| `supermarket.csv` | Main dataset — one row per customer, sequence of `dept · time · price` triplets | ~15% confirmed fraud |
| `cashier_data.csv` | Cashier-scanned transactions — items registered at checkout by staff | 100% clean (fraud-free) |
| `shrinkage.csv` | Aggregate per-department inventory audit — items and revenue expected vs registered | External ground truth |
| `stockage_missing.csv` | Individual missing-item records — department and confirmed unregistered item value | External ground truth |

---

### Notebook Structure

The analysis is organised into the following sections:

> Sections 1–10 focus exclusively on the self-scanner population: parsing,
> feature engineering, temporal patterns, distance behaviour, and anomaly
> scoring.

| Section | Content |
|---|---|
| 1 | Store Layout Construction |
| 2 | Data Parsing & Feature Engineering |
| 3 | Scan & Trip Overview |
| 4 | Time Beyween Scans Analysis |
| 5 | Department Dwell-Time Analysis |
| 6 | Travel Distance Analysis |
| 7 | Scan Behaviour Analysis |
| 8 | Cashier dataset comparison |
| 9 | Actual Missing Items Analysis (`shrinkage.csv` · `stockage_missing.csv`) |
| 10 | Distance Anomaly Scoring |

> **Note:** All monetary values are expressed in **€ (EUR)**. Time values
> are in **seconds**. Distance values are computed from the store's
> department layout grid and expressed in **arbitrary grid units**. Data is cleaned from zero-price scans.

## Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from collections import Counter, defaultdict
import warnings
from scipy.stats import zscore
warnings.filterwarnings('ignore')

TEAL      = '#01696f'
ORANGE    = '#da7101'
GOLD      = '#d19900'
TEAL_SOFT = '#4f98a3'
MUTED     = '#7a7974'
BG        = '#f9f8f5'

DEPT_MAP = {
    1: 'Bakery & Pastry',   2: 'Beer & Wine',       3: 'Books & Magazines',
    4: 'Candy & Chips',     5: 'Care & Hygiene',     6: 'Cereals & Spreads',
    7: 'Cheese & Tapas',    8: 'Dairy & Eggs',       9: 'Freezer',
    10: 'Fruit & Vegetables', 11: 'Household & Pet', 12: 'Meat & Fish',
    13: 'Pasta & Rice',     14: 'Salads & Meals',    15: 'Sauces & Spices',
    16: 'Soda & Juices',    17: 'Special Diet',      18: 'Vegetarian & Vegan',
}

DEPT_ABBR = {
     1:'BP',  2:'BW',  3:'BM',  4:'CC',  5:'CH',  6:'CS',
     7:'CT',  8:'DE',  9:'FR', 10:'FV', 11:'HP', 12:'MF',
    13:'PR', 14:'SM', 15:'SS', 16:'SJ', 17:'SD', 18:'VV',
}

GRID_LAYOUT = {
    'CI':(0,0), 'FV':(0,1), 'SM':(0,2), 'VV':(0,3), 'MF':(0,4),
    'DE':(1,0), 'SS':(1,1), 'PR':(1,2), 'CT':(1,3), 'BP':(1,4),
    'CS':(2,0), 'SD':(2,1), 'CC':(2,2), 'SJ':(2,3), 'FR':(2,4),
    'CO':(3,0), 'BM':(3,1), 'CH':(3,2), 'HP':(3,3), 'BW':(3,4),
}

DEPT_POS = {idx: GRID_LAYOUT[abbr]
            for idx, abbr in DEPT_ABBR.items()
            if abbr in GRID_LAYOUT}
DEPT_IDS = sorted(DEPT_POS.keys())

DIST_MATRIX = {
    (i, j): abs(DEPT_POS[i][0] - DEPT_POS[j][0]) +
             abs(DEPT_POS[i][1] - DEPT_POS[j][1])
    for i in DEPT_IDS for j in DEPT_IDS
}
print(f'✓ Distance matrix: {len(DIST_MATRIX)} pairs')
print(f'  Max possible distance: {max(DIST_MATRIX.values()):.3f} grid units')

---
## 1. Store Layout — Department Grid & Distance Matrix

The physical store is laid out on a **4-row × 5-column grid**:

```
Col:   0    1    2    3    4
Row 0: CI   FV   SM   VV   MF
Row 1: DE   SS   PR   CT   BP
Row 2: CS   SD   CC   SJ   FR
Row 3: CO   BM   CH   HP   BW
```

`CI` = Check-in, `CO` = Check-out (structural anchors — not scannable).  
All 18 scannable departments are encoded with `(row, col)` positions,  
and **all 18×18 pairwise Manhattan distances** are precomputed for O(1) lookup.


In [ ]:
# Store Layout Grid ───────────────────────────────────────────────
fig = go.Figure()

for abbr, (row, col) in GRID_LAYOUT.items():
    dept_id   = next((k for k, v in DEPT_ABBR.items() if v == abbr), None)
    is_scan   = dept_id is not None
    full_name = DEPT_MAP.get(dept_id, abbr) if is_scan else abbr
    cell_col  = TEAL if is_scan else MUTED
    hover_txt = f'<b>{abbr}</b><br>ID: {dept_id}<br>{full_name}' if is_scan else f'<b>{abbr}</b><br>(structural)'

    fig.add_shape(
        type='rect',
        x0=col - 0.45, y0=-(row + 0.45),
        x1=col + 0.45, y1=-(row - 0.45),
        fillcolor=cell_col, opacity=0.18,
        line=dict(color=cell_col, width=1.5),
    )
    label = f'{abbr}<br>({dept_id})' if is_scan else abbr
    fig.add_trace(go.Scatter(
        x=[col], y=[-row], mode='text',
        text=[label],
        textfont=dict(size=11, color=cell_col),
        hovertext=[hover_txt], hoverinfo='text',
        showlegend=False,
    ))

fig.update_layout(
    title=dict(text='Store Department Layout', font=dict(size=16, color='#28251d'), x=0.5),
    xaxis=dict(tickvals=list(range(5)), ticktext=[f'Col {i}' for i in range(5)],
               range=[-0.6, 4.6], showgrid=True, gridcolor='#e0e0e0', zeroline=False),
    yaxis=dict(tickvals=[-r for r in range(4)], ticktext=[f'Row {r}' for r in range(4)],
               range=[-3.6, 0.6], showgrid=True, gridcolor='#e0e0e0', zeroline=False,
               scaleanchor='x', scaleratio=1),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=700, height=400,
    margin=dict(l=60, r=30, t=60, b=40),
)
fig.show()


In [ ]:
# Distance Matrix Heatmap ────────────────────────────────────────
ids    = DEPT_IDS
labels = [f'{DEPT_ABBR[i]}<br>({i})' for i in ids]
mat    = np.array([[DIST_MATRIX[(i, j)] for j in ids] for i in ids], dtype=float)

fig = go.Figure(go.Heatmap(
    z=mat, x=labels, y=labels,
    colorscale=[[0, BG], [0.33, TEAL_SOFT], [0.66, TEAL], [1, '#003b3e']],
    text=mat.astype(int), texttemplate='%{text}',
    hovertemplate='From %{y} → To %{x}<br>Distance: %{z} grid units<extra></extra>',
    colorbar=dict(title='Grid units', thickness=14),
))
fig.update_layout(
    title=dict(text='Manhattan Distance Matrix (18 × 18 Departments)',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(tickfont=dict(size=9)),
    yaxis=dict(tickfont=dict(size=9), autorange='reversed'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=800, height=640,
    margin=dict(l=90, r=40, t=60, b=90),
)
fig.show()


---
## 2. Data Parsing & Feature Engineering

The raw CSV has no header. Each row is one customer; each comma-separated token  
is a scan in the format `dept time_sec price`.

We parse the file line-by-line and build two artefacts:
- **`df`** — one row per customer, 24 engineered features (saved to `customer_features.csv`).
- **`steps`** — one row per department-to-department step (used in distance analysis).


In [ ]:
customers = [] 
with open('supermarket.csv', 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        scans = []
        for token in line.split(','):
            parts = token.strip().split()
            if len(parts) == 3:
                try:
                    dept = int(parts[0])
                    t    = int(parts[1])
                    amt  = float(parts[2])
                    if amt > 0.0:        
                        scans.append((dept, t, amt))
                except ValueError:
                    pass
        if scans:
            customers.append(scans)

print(f'✓ Customers parsed: {len(customers):,}')

# ── Feature engineering — 24 features per customer ───────────────────────────
def compute_features(cust_id, scans):
    n       = len(scans)
    depts   = [s[0] for s in scans]
    times   = [s[1] for s in scans]
    amounts = [s[2] for s in scans]

    total_amount = sum(amounts)
    total_time   = sum(times)

    prev = depts[0]
    step_dists = []
    for i in range(1, n):
        curr = depts[i]
        if prev in DEPT_POS and curr in DEPT_POS:
            step_dists.append(DIST_MATRIX[(prev, curr)])
        prev = curr

    total_dist   = sum(step_dists)
    avg_dist     = float(np.mean(step_dists))   if step_dists else 0.0
    max_dist     = max(step_dists)               if step_dists else 0.0
    std_dist     = float(np.std(step_dists))     if len(step_dists) > 1 else 0.0
    n_large_jumps = sum(1 for d in step_dists if d > 3.0)
    n_zero_moves  = sum(1 for d in step_dists if d == 0.0)
    jump_ratio    = n_large_jumps / len(step_dists) if step_dists else 0.0

    dist_per_sec     = total_dist / total_time if total_time > 0 else 0.0
    unique_depts     = len(set(depts))
    fast_scans       = sum(1 for t in times if t < 10)
    long_dwells      = sum(1 for t in times if t > 600)
    low_amount_scans = sum(1 for a in amounts if a < 0.5)
    zero_amounts     = sum(1 for a in amounts if a == 0.0)
    high_value_scans = sum(1 for a in amounts if a > 10)
    fast_high_value  = sum(1 for t, a in zip(times, amounts) if t < 15 and a > 8)

    return {
        'cust_id':        cust_id,
        'n_scans':        n,
        'total_amount':   round(total_amount, 2),
        'avg_amount':     round(total_amount / n, 3),
        'total_time':     total_time,
        'avg_time':       round(total_time / n, 1),
        'unique_depts':   unique_depts,
        'dept_revisits':  n - unique_depts,
        'total_dist':     round(total_dist, 3),
        'avg_dist':       round(avg_dist, 3),
        'max_dist':       round(max_dist, 3),
        'std_dist':       round(std_dist, 3),
        'n_large_jumps':  n_large_jumps,
        'n_zero_moves':   n_zero_moves,
        'jump_ratio':     round(jump_ratio, 4),
        'dist_per_sec':   round(dist_per_sec, 6),
        'fast_scans':     fast_scans,
        'fast_scan_ratio': round(fast_scans / n, 4),
        'long_dwells':    long_dwells,
        'low_amount_ratio': round(low_amount_scans / n, 4),
        'zero_amounts':   zero_amounts,
        'high_value_scans': high_value_scans,
        'fast_high_value': fast_high_value,
        'amt_per_sec':    round(total_amount / total_time, 5) if total_time > 0 else 0.0,
    }

records = [compute_features(i, c) for i, c in enumerate(customers)]
df = pd.DataFrame(records)
df.to_csv('customer_features.csv', index=False)
print(f'✓ Feature matrix: {df.shape}  →  saved to customer_features.csv')
df.head(3)


### 24 Features per Customer explained

| Feature | Definition |
|---|---|
| `cust_id` | Unique customer index (row number in the CSV) |
| `n_scans` | Total number of scan events in the session |
| `total_amount` | Sum of all scanned item amounts (€) in the session |
| `avg_amount` | Average amount per scan event (`total_amount / n_scans`) |
| `total_time` | Sum of all dwell times across scan events (seconds) |
| `avg_time` | Average dwell time per scan event (`total_time / n_scans`) |
| `unique_depts` | Number of distinct departments visited |
| `dept_revisits` | Number of times a previously visited department was scanned again (`n_scans - unique_depts`) |

#### Travel Distance Features

| Feature | Definition |
|---|---|
| `total_dist` | Total path length: sum of all step distances across the session (grid units) |
| `avg_dist` | Average distance per step (`total_dist / n_steps`) |
| `max_dist` | Longest single step taken in the session (grid units) |
| `std_dist` | Standard deviation of step distances. Measures consistency of movement; high value = erratic path |
| `n_large_jumps` | Count of steps where distance > 3.0 grid units (cross-store jumps) |
| `n_zero_moves` | Count of steps where the customer scanned the same department twice in a row (distance = 0) |
| `jump_ratio` | Fraction of all steps that are large jumps (`n_large_jumps / n_steps`) |
| `dist_per_sec` | Implied movement speed. Total distance divided by total session time (`total_dist / total_time`) |

#### Scan Behaviour Features

| Feature | Definition |
|---|---|
| `fast_scans` | Count of scan events with dwell time < 10 seconds |
| `fast_scan_ratio` | Fraction of all scans that are fast (`fast_scans / n_scans`) |
| `long_dwells` | Count of scan events with dwell time > 600 seconds (10 min+; possible distraction or evasion) |
| `low_amount_ratio` | Fraction of scans with amount < €0.50 (cheap items or potential under-scanning) |
| `zero_amounts` | Count of scans with amount = €0.00 (items scanned but not charged) |
| `high_value_scans` | Count of scans with amount > €10 (expensive items) |
| `fast_high_value` | Count of scans where dwell time < 15s **and** amount > €8: expensive items scanned suspiciously fast|
| `amt_per_sec` | Scanning throughout the session. Total amount charged per second of session time (`total_amount / total_time`) |

---

**Thresholds used** (all adjustable):

| Threshold | Value | Description |
|---|---|---|
| Large jump | > 3.0 grid units | Covers ~75% of the store width diagonally |
| Fast scan | < 10 seconds | Below typical scan + bag time |
| Long dwell | > 600 seconds | 10 minutes in one spot |
| Low amount | < €0.50 | Below typical grocery item price |
| High value | > €10 | Above typical grocery item price |
| Fast + high value | time < 15s AND amount > €8 | Combined signal for deliberate under-scanning |


In [ ]:
# ── Step-level dataset ────────────────────────────────────────────────────────
step_rows = []
for cust_id, scans in enumerate(customers):
    prev = scans[0][0]
    for i in range(1, len(scans)):
        curr = scans[i][0]
        if prev in DEPT_POS and curr in DEPT_POS:
            step_rows.append((cust_id, prev, curr,
                               DIST_MATRIX[(prev, curr)],
                               scans[i][1], scans[i][2]))
        prev = curr

steps = pd.DataFrame(step_rows,
                     columns=['cust_id','from_dept','to_dept',
                              'distance','time_sec','amount'])
print(f'✓ Step transitions: {len(steps):,}')
print(f'  Distance range: {steps.distance.min():.2f} – {steps.distance.max():.2f} grid units')
steps.head(3)


### Step-Level Dataset — `steps` DataFrame

Unlike `customer_features.csv` (one row per customer), the `steps` DataFrame focuses on inspecting each customer's step in their journey

| Column | Meaning | Example |
|---|---|---|
| `cust_id` | Unique customer index — links back to `customer_features.csv` | `4821` |
| `from_dept` | Department ID the customer came **from** | `10` (FV — Fruit & Vegetables) |
| `to_dept` | Department ID the customer moved **to** | `12` (MF — Meat & Fish) |
| `distance` | Manhattan distance between the two departments (grid units) | `2.236` |
| `time_sec` | Dwell time spent at the **destination** department (seconds) | `45` |
| `amount` | Amount scanned at the **destination** department (€) | `8.99` |


In [ ]:
# Feature summary statistics ─────────────────────────────────────
summary_cols = ['n_scans','total_amount','avg_amount','total_time','avg_time',
                'unique_depts','dept_revisits','total_dist','avg_dist',
                'jump_ratio','fast_scan_ratio','low_amount_ratio','zero_amounts',
                'fast_high_value','amt_per_sec']

desc = df[summary_cols].describe().T.reset_index().rename(columns={'index':'feature'})
desc = desc.round(3)

fig = go.Figure(go.Table(
    columnwidth=[180,60,70,70,70,70,70,70,70],
    header=dict(
        values=['<b>Feature</b>','<b>count</b>','<b>mean</b>','<b>std</b>',
                '<b>min</b>','<b>25 %</b>','<b>50 %</b>','<b>75 %</b>','<b>max</b>'],
        fill_color=TEAL, font=dict(color='white', size=12),
        align='left', height=28,
    ),
    cells=dict(
        values=[desc[c] for c in ['feature','count','mean','std','min','25%','50%','75%','max']],
        fill_color=[[BG if i % 2 == 0 else 'white' for i in range(len(desc))]]*9,
        font=dict(color='#28251d', size=11),
        align='left', height=24,
    ),
))
fig.update_layout(
    title=dict(text='Feature Summary Statistics (100,000 customers)',
               font=dict(size=15, color='#28251d'), x=0.5),
    paper_bgcolor='white',
    margin=dict(l=20, r=20, t=50, b=20),
    width=900, height=520,
)
fig.show()


---
## 3. Scan & Trip Overview

Basic customer-level statistics: how many items they scan and how long they spend in store.


In [ ]:
# Scans per Customer ─────────────────────────────────────────────
scans_count = df['n_scans'].value_counts().sort_index()

fig = go.Figure(go.Bar(
    x=scans_count.index,
    y=scans_count.values,
    marker_color=TEAL,
    opacity=0.85,
    hovertemplate='Scans: %{x}<br>Customers: %{y:,}<extra></extra>',
))
fig.update_layout(
    title=dict(text='Distribution of Scans per Customer',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Number of Scans', tick0=0, dtick=5, tickfont=dict(size=10)),
    yaxis=dict(title='Number of Customers', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=420,
    margin=dict(l=70, r=30, t=60, b=60),
)
fig.show()

print(f'\nMost common scan count : {scans_count.idxmax()} scans ({scans_count.max():,} customers)')
print(f'Mean scans per customer : {df["n_scans"].mean():.1f}')
print(f'Max scans in one trip   : {df["n_scans"].max()}')


### Distribution of Scans per Customer Analysis

The histogram shows the frequency distribution of the number of scan events
per customer across all 100,000 shopping trips.

**Key observations:**

- The distribution is **right-skewed** with a clear unimodal peak around
  **15–17 scans**, where approximately 4,800–4,900 customers are concentrated.
- The bulk of customers (roughly 80 %) fall in the range of **8 to 35 scans**,
  representing a typical self-scan grocery trip.
- The distribution has a **long right tail** extending past 50 scans, with very
  few customers scanning more than 60 items. The maximum observed is 108 scans,
  though such extreme values are rare.
- The left boundary starts sharply at 3 scans: customers with fewer than 3
  valid scan events were excluded during parsing.


In [ ]:
# Trip Duration Distribution ─────────────────────────────────────
trip_min = df['total_time'] / 60

fig = go.Figure(go.Histogram(
    x=trip_min,
    nbinsx=80,
    marker_color=TEAL,
    opacity=0.82,
    hovertemplate='Duration: %{x:.0f} min<br>Count: %{y:,}<extra></extra>',
))

p95 = trip_min.quantile(0.95)
fig.add_vline(x=p95, line_dash='dash', line_color=ORANGE,
              annotation_text=f'95th pct: {p95:.0f} min',
              annotation_position='top right',
              annotation_font_color=ORANGE)

fig.update_layout(
    title=dict(text='Trip Duration per Customer (minutes)',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Trip Duration (min)', range=[0, trip_min.quantile(0.995)]),
    yaxis=dict(title='Number of Customers', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=420,
    margin=dict(l=70, r=30, t=60, b=60),
)
fig.show()

print(f'\nMean trip duration : {trip_min.mean():.1f} min')
print(f'Median trip duration: {trip_min.median():.1f} min')
print(f'95th percentile     : {p95:.1f} min')


### Trip Duration per Customer Analysis

The histogram shows the distribution of total shopping trip duration across
100000 customers, measured in minutes. Duration is computed as the sum
of all `time_sec` values within a customer's session.

**Key observations:**

- The distribution is **right-skewed** with a dominant peak at approximately
  **25–30 minutes**, where around 14,000 customers are concentrated. This is main "standard" grocery trip length.
- The majority of customers (roughly 80 %) complete their trip within
  **15 to 55 minutes**, which aligns well with typical self-scan shopping
  behaviour.
- The **95th percentile sits at 64 minutes** (orange dashed line), meaning
  only 5% of customers spend more than ~1 hour in the store.
- The tail extends beyond 90 minutes, though these extreme durations are
  very rare (< 1 % of customers).
- The distribution rises steeply from near-zero at ~5 minutes, reflecting
  the parsing filter that requires a minimum number of valid scans.

In [ ]:
# Scans vs Total Time scatter (subsample for speed) ───────────────
sample = df.sample(n=min(50000, len(df)), random_state=42)

fig = go.Figure(go.Scatter(
    x=sample['n_scans'],
    y=sample['total_time'] / 60,
    mode='markers',
    marker=dict(color=TEAL_SOFT, opacity=0.35, size=4),
    hovertemplate='Scans: %{x}<br>Duration: %{y:.1f} min<extra></extra>',
))
fig.update_layout(
    title=dict(text='Number of Scans vs Trip Duration (15,000-customer sample)',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Number of Scans', gridcolor='#e8e8e5'),
    yaxis=dict(title='Trip Duration (min)', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=430,
    margin=dict(l=70, r=30, t=60, b=60),
)
fig.show()


### Number of Scans vs Trip Duration Analysis (50000-customer sample) 

The scatter plot shows the relationship between the total number of scan
events and total trip duration (in minutes) for a random sample of 50000
customers, illustrating how shopping time scales with basket size.

**Key observations:**

- There is a **clear positive correlation** between scan count and trip
  duration — customers who scan more items generally spend more time in the
  store, which is the expected behaviour for honest shoppers.
- The relationship is **linear in its central tendency** but the spread
  increases substantially with scan count, forming a characteristic
  **fan-shaped (heteroscedastic) pattern**. At 40 scans, trip durations
  range roughly from 40 to 100 minutes; at 10 scans, the range is only
  5 to 35 minutes.
- The **lower envelope** of the cloud (fast trips relative to scan count)
  is the most fraud-relevant region. These customers completed many scans
  in a disproportionately short time.
- Outliers above the main cloud (very long trips for a given scan count)
  correspond to customers with one or more `long_dwells` events (> 600 s
  in a single department), which inflates total time without adding scans.
- A small number of extreme outliers are visible at 50–110 scans with
  durations exceeding 120 minutes, suggesting either very large baskets or
  sessions with multiple prolonged pauses.

---
## 4. Time Between Scans

Each `time_sec` field (after the first scan) represents the time elapsed since  
the previous scan — capturing both travel time and dwell time at the current department.  
Very short gaps may indicate rushed scanning; very long gaps suggest extended loitering.


In [ ]:
# Build long-format per-scan dataframe
all_scans = []
for cust_id, scans in enumerate(customers):
    for scan_number, (dept, t, amt) in enumerate(scans, start=1):
        all_scans.append({'cust_id': cust_id, 'scan_number': scan_number,
                          'department': dept, 'time_seconds': t, 'price': amt})
long = pd.DataFrame(all_scans)

upper_bound = long['time_seconds'].quantile(0.99)
print(f'Suspicious gap threshold (99th pct): {upper_bound:.0f} seconds')
print(f'Total scan events: {len(long):,}')

customer_stats = (
    long.groupby('cust_id')
    .agg(
        n_scans=('time_seconds','count'),
        total_time_sec=('time_seconds','sum'),
        avg_gap=('time_seconds','mean'),
        max_gap=('time_seconds','max'),
        n_suspicious_gaps=('time_seconds', lambda x: (x > upper_bound).sum()),
    )
    .reset_index()
)

long_sorted = long.sort_values(['cust_id','scan_number'])
long_sorted['prev_dept'] = long_sorted.groupby('cust_id')['department'].shift(1)
same_dept_count = int((
    long_sorted['prev_dept'].notna() &
    (long_sorted['department'] == long_sorted['prev_dept'])
).sum())
print(f'Same-department consecutive scans: {same_dept_count:,}')


In [ ]:
# Distribution of time between scans (≤ 600 s) ───────────────────
gap_data = long['time_seconds']
gap_clip = gap_data[gap_data <= 600]

fig = go.Figure(go.Histogram(
    x=gap_clip,
    nbinsx=100,
    marker_color=TEAL,
    opacity=0.82,
    hovertemplate='Gap: %{x} s<br>Count: %{y:,}<extra></extra>',
))
fig.add_vline(x=upper_bound, line_dash='dash', line_color=ORANGE,
              annotation_text=f'99th pct: {upper_bound:.0f} s',
              annotation_position='top right',
              annotation_font_color=ORANGE)
fig.add_vline(x=10, line_dash='dot', line_color=TEAL_SOFT,
              annotation_text='Fast-scan threshold: 10 s',
              annotation_position='top left',
              annotation_font_color=TEAL_SOFT)

fig.update_layout(
    title=dict(text='Time Between Scans (≤ 600 s shown)',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Seconds', gridcolor='#e8e8e5'),
    yaxis=dict(title='Scan Count', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=430,
    margin=dict(l=70, r=30, t=60, b=60),
)
fig.show()


### Time Between Scans Analysis (clipped at 600 seconds)

The histogram shows the distribution of inter-scan time gaps across all
~2 million individual scan events in the dataset (clipped at 600 s for
readability). Each value represents the time elapsed — combining travel
time and dwell time — between two consecutive scans for the same customer.

**Key observations:**

- The distribution is **strongly right-skewed** with a sharp peak at
  approximately **30–50 seconds**, representing the most common combined
  travel-and-dwell time between two consecutive department scans.
- Scan counts drop off rapidly after ~100 s and taper to near-zero by
  ~400 s, forming a classic exponential-decay shape typical of inter-event
  time distributions.
- The **fast-scan threshold at 10 seconds** (red dotted line) marks the
  left tail: any scan event with a gap below this value is flagged as a
  `fast_scan`. A non-trivial number of scans fall to the left of this line,
  visible as the small bar cluster between 0 and 10 s. Note the x-axis
  extends slightly into negative territory, which reflects a small number
  of data artefacts that are filtered during feature engineering.
- The **99th percentile sits at 581 seconds** (~9.7 min, orange dashed
  line). Gaps exceeding this threshold are classified as `suspicious_gaps`. This indicates customers who paused unusually long between scans,
  possibly to conceal items or wait for surveillance blind spots.

In [ ]:
# Suspicious gaps per customer ────────────────────────────────────
susp_counts = customer_stats['n_suspicious_gaps'].value_counts().sort_index()
susp_counts = susp_counts[susp_counts.index <= 15]

fig = go.Figure(go.Bar(
    x=susp_counts.index,
    y=susp_counts.values,
    marker_color=[RED if x >= 3 else TEAL for x in susp_counts.index],
    opacity=0.88,
    hovertemplate='Suspicious gaps: %{x}<br>Customers: %{y:,}<extra></extra>',
))
fig.add_vline(x=2.5, line_dash='dash', line_color=RED,
              annotation_text='≥ 3 suspicious gaps', annotation_font_color=RED)
fig.update_layout(
    title=dict(text='Number of Suspicious Gaps (> 99th pct) per Customer',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Number of Suspicious Gaps', tick0=0, dtick=1),
    yaxis=dict(title='Number of Customers', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=420,
    margin=dict(l=70, r=30, t=60, b=60),
)
fig.show()

pct_suspicious = (customer_stats['n_suspicious_gaps'] >= 3).mean() * 100
print(f'Customers with ≥ 3 suspicious gaps: {pct_suspicious:.1f} %')


In [ ]:
# Average gap vs max gap scatter ─────────────────────────────────
sample_cs = customer_stats.sample(n=min(50000, len(customer_stats)), random_state=1)

fig = go.Figure(go.Scatter(
    x=sample_cs['avg_gap'],
    y=sample_cs['max_gap'],
    mode='markers',
    marker=dict(color=TEAL_SOFT, size=4, opacity=0.35),
    hovertemplate='Avg gap: %{x:.0f} s<br>Max gap: %{y:.0f} s<extra></extra>',
))
fig.add_hline(y=upper_bound, line_dash='dash', line_color=ORANGE,
              annotation_text=f'99th pct threshold: {upper_bound:.0f} s',
              annotation_font_color=ORANGE)
fig.update_layout(
    title=dict(text='Average vs Maximum Inter-Scan Gap per Customer (sample)',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Average Gap (s)', gridcolor='#e8e8e5'),
    yaxis=dict(title='Maximum Gap (s)', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=430,
    margin=dict(l=70, r=30, t=60, b=60),
)
fig.show()


### Average vs Maximum Inter-Scan Gap per Customer Analysis

The scatter plot compares two per-customer timing aggregates (the average
inter-scan gap and the single longest gap recorded during their trip) for
a random sample of 50000 customers. Each point represents one customer.

**Key observations:**

- The cloud forms a **broad triangular shape**: customers with a low average
  gap (fast overall pace) cluster tightly near the origin, while customers
  with a higher average gap show much greater variance in their maximum gap.
  This is geometrically expected: the maximum can never fall below the
  average, creating a hard lower-bound diagonal.
- The **densest region** lies between 40–150 s on the x-axis and 100–600 s
  on the y-axis, representing the typical customer: moderate pacing with
  one or two longer pauses (for example, browsing a shelf or waiting at a counter).
- The **99th percentile threshold at 581 s** (orange dashed line) cleanly
  separates the majority of customers below from a meaningful cluster of
  outliers above. Customers above this line had at least one gap exceeding
  ~10 minutes. It is the key input to the `n_suspicious_gaps` feature.
- A distinct **upper-right quadrant** is visible: customers with both a
  high average gap (> 300 s) and a high maximum gap (> 581 s). These are
  the most time-anomalous customers in the dataset and are strong candidates
  for elevated `time_anomaly_score` values in Section 9.
- Several isolated extreme outliers reach maximum gaps of 1,500 s (~25 min),
  well above the bulk of the distribution.

---
## 5. Department Dwell-Time Analysis

We group all scan events by department and examine the distribution of time spent (travel + dwell) per scan. Departments with systematically shorter dwell times may indicate rushed scanning, which is a potential fraud signal.


In [ ]:
# Median dwell time per department ────────────────────────────────
dept_time = (
    long.groupby('department')['time_seconds']
    .agg(median='median', mean='mean', q75=lambda x: x.quantile(0.75))
    .reset_index()
)
dept_time['dept_name'] = dept_time['department'].map(DEPT_MAP)
dept_time = dept_time.sort_values('median', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=dept_time['dept_name'],
    x=dept_time['median'],
    orientation='h',
    name='Median',
    marker_color=TEAL,
    opacity=0.85,
    hovertemplate='%{y}<br>Median: %{x:.0f} s<extra></extra>',
))
fig.add_trace(go.Scatter(
    y=dept_time['dept_name'],
    x=dept_time['mean'],
    mode='markers',
    name='Mean',
    marker=dict(color=ORANGE, size=9, symbol='circle'),
    hovertemplate='%{y}<br>Mean: %{x:.0f} s<extra></extra>',
))
fig.update_layout(
    title=dict(text='Median & Mean Dwell Time per Department',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Time (seconds)', gridcolor='#e8e8e5'),
    yaxis=dict(title='', tickfont=dict(size=11)),
    legend=dict(orientation='h', yanchor='bottom', y=1.01, x=0.6),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=540,
    margin=dict(l=180, r=30, t=70, b=60),
)
fig.show()


### Median & Mean Dwell Time per Department

The chart shows the median (teal bar) and mean (orange dot) time spent per
scan event for each of the 18 scannable departments, sorted by ascending
median dwell time. Both metrics combine travel time from the previous
department and dwell time at the current one.

**Key observations:**

- **Freezer** has the highest median dwell time (~97 s), followed closely
  by **Beer & Wine** (~90 s) and **Pasta & Rice** (~82 s). These
  departments tend to require more deliberation — customers compare
  products, check prices, or spend time loading heavy/bulky items into
  their basket.
- **Special Diet** (~37 s), **Fruit & Vegetables** (~43 s), and
  **Cheese & Tapas** (~48 s) occupy the lowest end of the ranking,
  suggesting faster, more decisive shopping behaviour in these areas.
- The **mean consistently exceeds the median** for every department:
  visible as orange dots always sitting to the right of the teal bars.
  This confirms that dwell-time distributions are right-skewed at the
  department level, pulled upward by a minority of customers who linger
  exceptionally long.
- The **gap between median and mean is most pronounced** for **Beer &
  Wine** (~90 s median vs ~180 s mean), **Freezer** (~97 s median vs
  ~155 s mean), and **Household & Pet** and **Care & Hygiene**
  (both showing mean values around 145–155 s). These large divergences
  indicate a meaningful tail of customers spending an unusually long time
  in these departments, a pattern consistent with the `long_dwells`
  feature (> 600 s threshold).
- Departments such as **Fruit & Vegetables** and **Special Diet** show
  tight median–mean gaps, suggesting more homogeneous, predictable
  dwell behaviour with fewer extreme outliers.

**Fraud-detection relevance:**

Department-level dwell time norms provide the baseline for identifying
anomalous behaviour at a base level:
- A customer with a **very short dwell** (below median) in a
  high-attention department like Freezer or Beer & Wine (while also
  registering a high-value scan) is a candidate for the
  `fast_high_value` flag.
- Conversely, departments with high mean-median gaps (Beer & Wine,
  Freezer) are natural locations for `long_dwells` events, and an
  unusually long pause there combined with a low scanned amount
  warrants investigation.
- In later modelling sections, department-specific z-scoring of dwell
  time can complement the global `fast_scan_ratio` feature by accounting
  for the natural variation in expected time across departments.

In [ ]:
# Scan volume per department ─────────────────────────────────────
dept_vol = long['department'].value_counts().reset_index()
dept_vol.columns = ['department','scan_count']
dept_vol['dept_name'] = dept_vol['department'].map(DEPT_MAP)
dept_vol = dept_vol.sort_values('scan_count', ascending=True)

fig = go.Figure(go.Bar(
    y=dept_vol['dept_name'],
    x=dept_vol['scan_count'],
    orientation='h',
    marker_color=TEAL,
    opacity=0.88,
    hovertemplate='%{y}<br>Scans: %{x:,}<extra></extra>',
))
fig.update_layout(
    title=dict(text='Total Scan Volume per Department',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Number of Scans', gridcolor='#e8e8e5'),
    yaxis=dict(title='', tickfont=dict(size=11)),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=520,
    margin=dict(l=180, r=30, t=60, b=60),
)
fig.show()


In [ ]:
# Dwell time distribution per dept 
long_clip = long[long['time_seconds'] <= 600].copy()
long_clip['dept_name'] = long_clip['department'].map(DEPT_MAP)

dept_order = (
    long_clip.groupby('dept_name')['time_seconds']
    .median().sort_values().index.tolist()
)

fig = go.Figure()
for dept in dept_order:
    sub = long_clip[long_clip['dept_name'] == dept]['time_seconds']
    fig.add_trace(go.Box(
        x=sub, name=dept,
        marker_color=TEAL,
        line_color=TEAL,
        fillcolor=TEAL_SOFT,
        opacity=0.65,
        boxpoints=False,
        orientation='h',
        hovertemplate=f'{dept}<br>%{{x}} s<extra></extra>',
    ))

fig.update_layout(
    title=dict(text='Dwell-Time Distribution per Department (≤ 600 s)',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Time (seconds)', gridcolor='#e8e8e5'),
    yaxis=dict(tickfont=dict(size=10)),
    showlegend=False,
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=600,
    margin=dict(l=180, r=30, t=60, b=60),
)
fig.show()


In [ ]:
# Dwell time per distance bucket ──────────────────────────────────
steps['dist_bucket'] = steps['distance'].round(1).astype(str)
order = sorted(steps['dist_bucket'].value_counts().head(10).index, key=float)
plot_data = steps[steps['dist_bucket'].isin(order)].copy()
plot_data['time_clipped'] = plot_data['time_sec'].clip(upper=600)

fig = px.box(
    plot_data.sort_values('dist_bucket', key=lambda s: s.astype(float)),
    x='dist_bucket', y='time_clipped',
    category_orders={'dist_bucket': order},
    labels={
        'dist_bucket' : 'Travel Distance (grid units)',
        'time_clipped': 'Time at Destination (s, clipped at 600)',
    },
    title='Dwell Time Distribution per Distance Bucket',
    color_discrete_sequence=[TEAL],
    points=False,
)
fig.update_layout(
    plot_bgcolor=BG, paper_bgcolor='white',
    width=750, height=420,
    margin=dict(l=70, r=30, t=60, b=60),
    title_font=dict(size=15, color='#28251d'), title_x=0.5,
    yaxis=dict(gridcolor='#e8e8e5', title='Time at Destination (s)'),
    xaxis=dict(title='Travel Distance (grid units)'),
)
fig.show()

---
## 6. Travel Distance Analysis

Using the Manhattan distance matrix from Section 1, each step between consecutive departments is assigned a physical distance. Cross-store jumps (distance > 3 grid units) are anomalous because they imply the customer teleported across the store — potentially scanning items they never physically approached.


In [ ]:
# Step-distance distribution ─────────────────────────────────────
fig = go.Figure(go.Histogram(
    x=steps['distance'],
    nbinsx=30,
    marker_color=TEAL,
    opacity=0.85,
    hovertemplate='Distance: %{x} grid units<br>Count: %{y:,}<extra></extra>',
))
fig.add_vline(x=3.0, line_dash='dash', line_color=ORANGE,
              annotation_text='Large-jump threshold: 3 units',
              annotation_position='top right',
              annotation_font_color=ORANGE)
fig.update_layout(
    title=dict(text='Distribution of Step Distances Between Departments',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Manhattan Distance (grid units)', gridcolor='#e8e8e5', dtick=1),
    yaxis=dict(title='Number of Steps', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=420,
    margin=dict(l=70, r=30, t=60, b=60),
)
fig.show()

pct_large = (steps['distance'] > 3).mean() * 100
pct_zero  = (steps['distance'] == 0).mean() * 100
print(f'Steps with distance > 3 (large jumps): {pct_large:.1f} %')
print(f'Steps with distance = 0 (same dept)   : {pct_zero:.1f} %')


In [ ]:
# Distance feature histograms ───────────────────────────────
def dist_histogram(col, title, color=TEAL):
    data     = df[col].dropna()
    mean_v   = data.mean()
    median_v = data.median()
    p95_v    = data.quantile(0.95)

    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=data, nbinsx=60,
        marker_color=color, opacity=0.78,
        name='Customers',
        hovertemplate='Value: %{x:.2f}<br>Count: %{y:,}<extra></extra>',
    ))

    for val, dash, lcolor in [
        (mean_v,   'dash',  '#28251d'),
        (median_v, 'dot',   MUTED),
        (p95_v,    'solid', ORANGE),
    ]:
        fig.add_vline(x=val, line_dash=dash, line_color=lcolor, line_width=1.8)

    stats_text = (
        f"<span style='color:#28251d'><b>Mean</b> {mean_v:.3f}</span>"
        f"&nbsp;&nbsp;&nbsp;"
        f"<span style='color:{MUTED}'><b>Median</b> {median_v:.3f}</span>"
        f"&nbsp;&nbsp;&nbsp;"
        f"<span style='color:{ORANGE}'><b>p95</b> {p95_v:.3f}</span>"
    )
    fig.add_annotation(
        text=stats_text,
        xref='paper', yref='paper',
        x=0.5, y=-0.25,
        showarrow=False,
        font=dict(size=12),
        align='center',
    )

    fig.update_layout(
        title=dict(text=title, font=dict(size=15, color='#28251d'), x=0.5),
        xaxis_title=col,
        yaxis_title='Number of Customers',
        yaxis=dict(gridcolor='#e8e8e5'),
        plot_bgcolor=BG,
        paper_bgcolor='white',
        bargap=0.04,
        width=750, height=400,
        margin=dict(l=60, r=30, t=60, b=80),
        showlegend=False,
    )
    fig.show()


dist_histogram('total_dist',    'Total Path Length per Customer (grid units)',                   TEAL)
dist_histogram('avg_dist',      'Average Step Distance per Customer (grid units)',               TEAL)
dist_histogram('max_dist',      'Maximum Single Step Distance per Customer',                     ORANGE)
dist_histogram('std_dist',      'Std Dev of Step Distances per Customer',                        MUTED)
dist_histogram('n_large_jumps', 'Number of Cross-Store Jumps (> 3 grid units) per Customer',    ORANGE)
dist_histogram('jump_ratio',    'Fraction of Steps That Are Cross-Store Jumps (> 3 units)',      ORANGE)

## Travel Distance Distribution Analysis - What do we observe

### Total Path Length per Customer 

The distribution is **right-skewed** with a clear peak around **12–15 grid units**, a mean of **14.082**
and a median of **13.000**. The fact that these two values are very similar shows that most customers move through the store in a simple and predictable way
The long right tail extending beyond 30–60 grid units indicates that a small subset of customers
have unusual, erratic paths, and the p95 threshold of **25.000** serves as a practical
upper boundary for "normal" shopping behaviour.

### Average Step Distance per Customer 

The distribution peaks at around **0.6–0.7 grid units** — meaning most customers
move between adjacent departments on each step, proven by mean of **0.807** and
median of **0.750**.
The p95 value of **1.333** shows that only 5% of customers average more than ~1.2 grid
units per step, suggesting that cross-department jumps are very rare and least favorite.

### Maximum Single Step Distance per Customer 

Unlike `total_dist` and `avg_dist`, this distribution is **discrete and multimodal**.
The largest cluster falls at **2.0–2.5 grid units** (mean 2.500, median 2.236),
meaning most customers' single longest step is a moderate cross-aisle move,
while the p95 of **4.000** flags customers whose worst jump spans a significant
portion of the store.

### Std Dev of Step Distances per Customer 

The distribution is **approximately normal** and the most symmetric of all six metrics.
It peaks at around **0.7–0.8 grid units** with a mean of **0.893** and median of **0.863**, which means that most customers have a consistent, moderately varied movement pattern.
Customers beyond the p95 threshold of **1.299** show high within-session variability.

### Number of Cross-Department Jumps per Customer

The distribution is **heavily zero-inflated** — alomost 80,000 customers (the majority)
make **zero** large cross-department jumps (> 3 grid units) during their entire session, suggesting that large-distance transitions are genuinely rare in normal shopping behaviour.
The p95 value of **1.000** means that only the top 5% of customers make even a single
large cross-department jump.

### Fraction of Steps That Are Cross-Store Jumps 

The distribution is **extremely right-skewed** with a median of **0.000**, showing that majority of customers have a `jump_ratio` of exactly zero, meaning they never make a large cross-department jump at all throughout their session.
95th percentile of **0.091** shows that only 5% of customers have more than ~8% of their steps classified as large jumps.

In [ ]:
# Most & least frequent department transitions ───────────────
cross = steps[steps['from_dept'] != steps['to_dept']].copy()

trans_flat = (
    cross.groupby(['from_dept', 'to_dept'])
         .agg(count=('cust_id', 'size'), avg_dist=('distance', 'first'))
         .reset_index()
         .sort_values('count', ascending=False)
)
trans_flat['label'] = (
    trans_flat['from_dept'].map(DEPT_MAP).str.split(' & ').str[0] + ' → ' +
    trans_flat['to_dept'].map(DEPT_MAP).str.split(' & ').str[0]
)

# ── Top 20 ────────────────────────────────────────────────────────────────────
top20 = trans_flat.head(20).sort_values('count')

fig_top = go.Figure(go.Bar(
    x=top20['count'], y=top20['label'],
    orientation='h',
    marker_color=ORANGE, opacity=0.85,
    hovertemplate='%{y}<br>Count: %{x:,}<extra></extra>',
))
fig_top.update_layout(
    title=dict(text='Top 20 Most Frequent Transitions',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Number of Transitions', gridcolor='#e8e8e5'),
    yaxis=dict(tickfont=dict(size=10)),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=750, height=520,
    margin=dict(l=200, r=30, t=60, b=50),
)
fig_top.show()

# ── Bottom 20 (non-zero) ──────────────────────────────────────────────────────
bot20 = (trans_flat[trans_flat['count'] > 0]
         .tail(20)
         .sort_values('count'))

fig_bot = go.Figure(go.Bar(
    x=bot20['count'], y=bot20['label'],
    orientation='h',
    marker_color=ORANGE, opacity=0.85,
    hovertemplate='%{y}<br>Count: %{x:,}<extra></extra>',
))
fig_bot.update_layout(
    title=dict(text='Bottom 20 Least Frequent Transitions (Non-Zero)',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Number of Transitions', gridcolor='#e8e8e5'),
    yaxis=dict(tickfont=dict(size=10)),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=750, height=520,
    margin=dict(l=200, r=30, t=60, b=50),
)
fig_bot.show()


---
## 7. Scan Behaviour Analysis

Fraud-relevant behavioural signals: fast scans, high-value items scanned quickly, zero-price items, and low-amount ratios.


In [ ]:
# Fast-scan ratio distribution ────────────────────────────────────
fig = go.Figure(go.Histogram(
    x=df['fast_scan_ratio'],
    nbinsx=50,
    marker_color=ORANGE,
    opacity=0.82,
    hovertemplate='Fast-scan ratio: %{x:.2f}<br>Customers: %{y:,}<extra></extra>',
))
fig.add_vline(x=df['fast_scan_ratio'].quantile(0.95), line_dash='dash', line_color=ORANGE,
              annotation_text='95th pct', annotation_font_color=ORANGE)
fig.update_layout(
    title=dict(text='Distribution of Fast-Scan Ratio per Customer  (scans < 10 s)',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Fast-Scan Ratio', gridcolor='#e8e8e5'),
    yaxis=dict(title='Number of Customers', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=850, height=420,
    margin=dict(l=70, r=30, t=60, b=60),
)
fig.show()


In [ ]:
# Price per scan distribution ────────────────────────────────────
price_sample = long['price'].sample(n=min(200000, len(long)), random_state=7)
price_clip   = price_sample[price_sample <= 30]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['All Prices ≤ €30', 'Low-Price Detail (≤ €3)'])

fig.add_trace(go.Histogram(
    x=price_clip, nbinsx=80,
    marker_color=TEAL, opacity=0.82,
    hovertemplate='Price: €%{x:.2f}<br>Count: %{y:,}<extra></extra>',
    name='All prices',
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=price_clip[price_clip <= 3], nbinsx=60,
    marker_color=ORANGE, opacity=0.82,
    hovertemplate='Price: €%{x:.2f}<br>Count: %{y:,}<extra></extra>',
    name='Low prices',
), row=1, col=2)

fig.add_vline(x=0.5, line_dash='dash', line_color=RED, row=1, col=2,
              annotation_text='Low-price threshold: €0.50', annotation_font_color=RED)

fig.update_layout(
    title=dict(text='Distribution of Scanned Item Prices (200 k scan sample)',
               font=dict(size=15, color='#28251d'), x=0.5),
    showlegend=False,
    plot_bgcolor=BG, paper_bgcolor='white',
    width=900, height=400,
    margin=dict(l=60, r=40, t=80, b=60),
)
fig.update_xaxes(title_text='Price (€)', gridcolor='#e8e8e5')
fig.update_yaxes(title_text='Count', gridcolor='#e8e8e5')
fig.show()


### Distribution of Scanned Item Prices Analysis (200000 scan sample)

A dual-panel histogram examining the price distribution of individual
scanned items across a 200000-scan random sample. The left panel shows
the full range up to €30; the right panel zooms into the low-price region
(≤ €3) with the `low_amount` threshold annotated.

**Left panel — All prices ≤ €30:**

- The distribution is **heavily right-skewed** with the vast majority of
  scanned items priced below €3, creating a sharp spike at the left edge
  that drops steeply and tapers toward zero by ~€10.
- The peak occurs at approximately **€0.50–€1.50**, reflecting the
  dominance of everyday grocery items (beverages, snacks, fresh produce)
  in the typical basket.
- Items priced above €10 are rare but present, which corresponds to premium
  products, alcohol, or speciality goods captured by the `high_value_scans`
  feature (price > €10).
- The distribution confirms a strong **floor effect near zero**: a
  non-trivial number of scan events register prices at or close to €0.00,
  which are tracked separately by the `zero_amounts` feature.

**Right panel — Low-price detail (≤ €3):**

- Zooming in reveals a **roughly bell-shaped sub-distribution** between
  €0 and €3, with a peak around **€1.10–€1.30** and a secondary
  concentration between €1.50 and €2.00.
- The **low-price threshold at €0.50** (red dashed line) cleanly separates
  a meaningful left-tail segment: items below this value are flagged as
  `low_amount` scans. The count in this region is substantial, indicating
  that many legitimate items (e.g. loose fruit, small confectionery) fall
  below the threshold, so `low_amount_ratio` must be interpreted alongside
  other signals rather than in isolation.
- The relatively flat right portion of the right panel (€2.00–€3.00)
  suggests a gradual transition into mid-range pricing without a sharp
  secondary peak.

**Fraud-detection relevance:**

The price distribution directly informs three engineered features:
- **`zero_amounts`**: scans at exactly €0.00, the strongest per-item
  price signal for deliberate non-charging.
- **`low_amount_ratio`**: fraction of scans below €0.50; elevated values
  combined with a high `fast_scan_ratio` suggest systematic substitution
  of expensive items with cheap ones at the scanner.
- **`high_value_scans`** and **`fast_high_value`**: items priced above
  €10 scanned in under 15 seconds represent the highest financial risk
  per individual scan event and are weighted accordingly in the anomaly
  scoring framework.

---
## 8. `cashier_data` — Comparative Analysis with `supermarket`

`cashier_data.csv` contains 100 % verified fraud-free transactions recorded at a traditional cashier checkout. Unlike `supermarket.csv`, each scan entry consists of only **two fields**: department ID and price (because the cashier controls the scanning pace and there is no customer travel time
between scans). Comparing the two datasets reveals the behavioural fingerprint of honest shopping, providing a clean reference baseline for the fraud
detection model.

In [ ]:
# Load and parse cashier_data.csv ────────────────────────────────
def parse_cashier(path):
    rows = []
    with open(path, 'r') as f:
        for cid, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            for pair in [p.strip() for p in line.split(',') if p.strip()]:
                parts = pair.split()
                if len(parts) == 2:
                    try:
                        rows.append({
                            'cust_id': cid,
                            'dept':    int(parts[0]),
                            'price':   float(parts[1]),
                        })
                    except ValueError:
                        pass
    return pd.DataFrame(rows)

cashier_scans = parse_cashier('cashier_data.csv')

cashier_cust = (cashier_scans.groupby('cust_id')
    .agg(
        n_scans      = ('price', 'size'),
        total_amount = ('price', 'sum'),
        zero_amounts = ('price', lambda x: (x == 0).sum()),
        n_depts      = ('dept',  'nunique'),
        avg_price    = ('price', 'mean'),
    )
    .reset_index()
)
cashier_cust['zero_ratio']   = cashier_cust['zero_amounts'] / cashier_cust['n_scans']
cashier_cust['amt_per_scan'] = cashier_cust['total_amount'] / cashier_cust['n_scans']

print(f"Cashier customers : {len(cashier_cust):,}")
print(f"Scan events       : {len(cashier_scans):,}")
cashier_cust.describe().round(3)

### Scan Count Distribution: Self-Scanner vs Cashier

The number of scans per customer compared across both datasets reveals whether the two populations shop in similarly-sized baskets, or whether basket size itself differs between the two checkout modes.

In [ ]:
# Scan count distribution. Self-scanner vs cashier ───────────────
ss_scans   = df['n_scans'].clip(upper=80)
cash_scans = cashier_cust['n_scans'].clip(upper=80)

fig = go.Figure()
for data, label, color in [
    (ss_scans,   'Self-Scanner (supermarket.csv)',  TEAL),
    (cash_scans, 'Cashier (cashier_data.csv)',       ORANGE),
]:
    fig.add_trace(go.Histogram(
        x=data, nbinsx=50, name=label,
        marker_color=color, opacity=0.65,
        hovertemplate='Scans: %{x}<br>Count: %{y:,}<extra></extra>',
    ))

fig.update_layout(
    barmode='overlay',
    title=dict(text='Scan Count Distribution: Self-Scanner vs Cashier', font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Number of Scans', gridcolor='#e8e8e5'),
    yaxis=dict(title='Number of Customers', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=750, height=420,
    margin=dict(l=60, r=30, t=60, b=60),
    legend=dict(x=0.65, y=0.95, bgcolor='rgba(0,0,0,0)'),
)
fig.show()

### Average Price per Scan: Self-Scanner vs Cashier

The mean price per individual scan event reflects basket composition. A lower average price in the self-scanner population could indicate selective scanning of cheaper items — a known fraud pattern.

In [ ]:
# Average price per scan ─────────────────────────────────────────
ss_avg   = df['total_amount'].div(df['n_scans']).clip(upper=8)
cash_avg = cashier_cust['amt_per_scan'].clip(upper=8)

fig = go.Figure()
for data, label, color in [
    (ss_avg,   'Self-Scanner', TEAL),
    (cash_avg, 'Cashier',      ORANGE),
]:
    fig.add_trace(go.Histogram(
        x=data, nbinsx=60, name=label,
        marker_color=color, opacity=0.65,
        hovertemplate='Avg price: €%{x:.2f}<br>Count: %{y:,}<extra></extra>',
    ))

fig.update_layout(
    barmode='overlay',
    title=dict(text='Average Price per Scan: Self-Scanner vs Cashier', font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Average Price per Scan (€, clipped at €8)', gridcolor='#e8e8e5'),
    yaxis=dict(title='Number of Customers', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=750, height=420,
    margin=dict(l=60, r=30, t=60, b=60),
    legend=dict(x=0.65, y=0.95, bgcolor='rgba(0,0,0,0)'),
)
fig.show()

### Average Price per Scan: Self-Scanner vs Cashier

This histogram compares the distribution of the **mean price per individual
scan event** between the two checkout modes.

**Key observations:**

- The **cashier distribution** (orange) is narrow and tightly concentrated
  between **€1 and €3**, peaking sharply around **€1.50–€2.00**. This is
  consistent with a verified fraud-free population where every item is
  scanned and charged at its true price.

- The **self-scanner distribution** (teal) is **much wider and bimodal**,
  with a first mode around €1.50–€2.00 and a long, heavy tail extending
  well past €5.00. The elevated right tail suggests a subset of customers
  registering disproportionately high-value items or, conversely, the
  low left-hand mass may reflect customers who scan many cheap items while
  skipping expensive ones.

- The self-scanner population is also **far larger** (~10× more customers),
  which is expected given the dataset sizes, but the shape difference is
  structural, not a sample-size artefact.

**Fraud relevance:** A systematically low average price per scan (below the
cashier baseline of €2.00) is a candidate feature for the fraud model. It
may indicate *selective scanning*, where a shopper scans a cheaper substitute
item while pocketing the higher-value product.

### Department Visit Frequency: Self-Scanner vs Cashier

Normalised scan volume per department (share of total scans) reveals whether self-scanner customers visit the same departments as cashier customers, or whether certain departments are systematically under-represented in self-scanner data — a potential proxy for avoidance of high-value zones.

In [ ]:
# Department scan share — self-scanner vs cashier ─────────────────
ss_dept_share   = (steps['to_dept'].value_counts(normalize=True)
                   .rename('self_scanner').rename_axis('dept').reset_index())
cash_dept_share = (cashier_scans['dept'].value_counts(normalize=True)
                   .rename('cashier').rename_axis('dept').reset_index())

dept_cmp = ss_dept_share.merge(cash_dept_share, on='dept', how='outer').fillna(0)
dept_cmp['dept_name'] = dept_cmp['dept'].map(DEPT_MAP)
dept_cmp['diff'] = dept_cmp['self_scanner'] - dept_cmp['cashier']
dept_cmp = dept_cmp.sort_values('diff')

fig = go.Figure()
fig.add_trace(go.Bar(
    y=dept_cmp['dept_name'], x=dept_cmp['self_scanner'],
    name='Self-Scanner', orientation='h',
    marker_color=TEAL, opacity=0.80,
))
fig.add_trace(go.Bar(
    y=dept_cmp['dept_name'], x=dept_cmp['cashier'],
    name='Cashier', orientation='h',
    marker_color=ORANGE, opacity=0.80,
))
fig.update_layout(
    barmode='group',
    title=dict(text='Department Scan Share: Self-Scanner vs Cashier', font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Share of Total Scans', tickformat='.1%', gridcolor='#e8e8e5'),
    yaxis=dict(tickfont=dict(size=10)),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=760, height=540,
    margin=dict(l=180, r=30, t=60, b=60),
    legend=dict(x=0.70, y=0.05, bgcolor='rgba(0,0,0,0)'),
)
fig.show()

### Total Basket Value Distribution: Self-Scanner vs Cashier

The distribution of total spend per customer session. If the self-scanner population systematically registers lower basket values than the cashier population for comparable scan counts, it corroborates the hypothesis of selective or partial scanning.

In [ ]:
# Total basket value distribution ─────────────────────────────────
ss_amt   = df['total_amount'].clip(upper=150)
cash_amt = cashier_cust['total_amount'].clip(upper=150)

fig = go.Figure()
for data, label, color in [
    (ss_amt,   'Self-Scanner', TEAL),
    (cash_amt, 'Cashier',      ORANGE),
]:
    fig.add_trace(go.Histogram(
        x=data, nbinsx=60, name=label,
        marker_color=color, opacity=0.65,
        hovertemplate='Basket: €%{x:.0f}<br>Count: %{y:,}<extra></extra>',
    ))

for val, label, ds_color in [
    (ss_amt.mean(),   'Self-Scanner', TEAL),
    (cash_amt.mean(), 'Cashier', ORANGE),
]:
    fig.add_vline(x=val, line_dash='dash', line_color=ds_color, line_width=1.8)

fig.update_layout(
    barmode='overlay',
    title=dict(text='Total Basket Value: Self-Scanner vs Cashier (clipped at €150)', font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Total Basket Value (€)', gridcolor='#e8e8e5'),
    yaxis=dict(title='Number of Customers', gridcolor='#e8e8e5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=750, height=430,
    margin=dict(l=60, r=30, t=60, b=60),
    legend=dict(x=0.68, y=0.95, bgcolor='rgba(0,0,0,0)'),
)
fig.show()

## 9. Actual Missing Items — What the Data Tells Us

Two ground-truth datasets from AHA:

| File | Content |
|---|---|
| `shrinkage.csv` | Aggregate per-department inventory audit: items expected vs registered, revenue expected vs registered, and derived shrinkage rates |
| `stockage_missing.csv` | Individual missing-item records with department and item value (€) — each row is one confirmed unregistered item |

Together they provide an **independent, audit-based signal** of where theft and scanning omissions actually occur in the store. Cross-referencing these departments against the self-scanner behavioural flags strengthens the fraud model's interpretability.

> **High-risk threshold:** departments with a revenue shrinkage rate ≥ 3 % are flagged in orange throughout this section.

In [ ]:
# Load shrinkage and missing-stock data ─────────────────────────

shrinkage = pd.read_csv('shrinkage.csv', sep=';', decimal=',').dropna(subset=['department'])
shrinkage['department'] = shrinkage['department'].astype(int)
shrinkage['dept_name']  = shrinkage['department'].map(DEPT_MAP)
shrinkage = shrinkage.sort_values('revenue_shrinkage', ascending=False).reset_index(drop=True)

# Individual missing-item records
missing = pd.read_csv('stockage_missing.csv', sep=';', decimal=',', usecols=[0, 1])
missing.columns = ['department', 'item_value']
missing = missing.dropna()
missing['department'] = pd.to_numeric(missing['department'], errors='coerce')
missing['item_value']  = pd.to_numeric(missing['item_value'],  errors='coerce')
missing = missing.dropna()
missing['department']  = missing['department'].astype(int)
missing = missing[missing['item_value'] > 0]

# High-risk set and shrinkage weights (used downstream by fraud model)
HIGH_RISK_DEPTS = set(shrinkage.loc[shrinkage['revenue_shrinkage'] >= 0.03, 'department'].tolist())
max_shrink      = shrinkage['revenue_shrinkage'].max()
SHRINK_WEIGHT   = dict(zip(shrinkage['department'],
                            shrinkage['revenue_shrinkage'] / max_shrink))

print(f'✓ Shrinkage loaded  : {len(shrinkage)} departments')
print(f'✓ Missing loaded    : {len(missing):,} records')
print(f'  High-risk depts (≥3% rev shrinkage): {sorted(HIGH_RISK_DEPTS)}')
print(f'  Total revenue lost : €{shrinkage["revenue_lost"].sum():,.2f}')
print(f'  Total items  lost  : {shrinkage["items_lost"].sum():,}')

In [ ]:
# Inventory Shrinkage Analysis ───────────────────────────────────
shr    = shrinkage.sort_values('revenue_shrinkage')
colors = [ORANGE if d in HIGH_RISK_DEPTS else TEAL for d in shr['department']]
labels = [f"{int(d)} {DEPT_MAP[d]}" for d in shr['department']]

fig_a = make_subplots(rows=1, cols=2,
    subplot_titles=['Revenue Shrinkage Rate (%) per Department',
                    'Total Revenue Lost (€) per Department'],
    column_widths=[0.48, 0.52], horizontal_spacing=0.14)

fig_a.add_trace(go.Bar(y=labels, x=shr['revenue_shrinkage'] * 100,
    orientation='h', marker_color=colors,
    text=[f'{v*100:.2f}%' for v in shr['revenue_shrinkage']],
    textposition='outside', textfont=dict(size=9), name='Rev shrinkage',
    hovertemplate='%{y}<br>Shrinkage: %{x:.3f}%<extra></extra>'), row=1, col=1)

fig_a.add_trace(go.Bar(y=labels, x=shr['revenue_lost'],
    orientation='h', marker_color=colors,
    text=[f'€{v:,.0f}' for v in shr['revenue_lost']],
    textposition='outside', textfont=dict(size=9), name='Revenue lost',
    hovertemplate='%{y}<br>Lost: €%{x:,.2f}<extra></extra>'), row=1, col=2)

fig_a.add_trace(go.Bar(y=[None], x=[None], name='High-risk ≥3%',
    marker_color=ORANGE, showlegend=True), row=1, col=1)
fig_a.add_trace(go.Bar(y=[None], x=[None], name='Low-risk <3%',
    marker_color=TEAL, showlegend=True), row=1, col=1)

fig_a.update_xaxes(gridcolor='#dcd9d5', tickfont=dict(size=9))
fig_a.update_yaxes(tickfont=dict(size=9))
fig_a.update_layout(
    title=dict(
        text='Inventory Shrinkage Analysis by Department<br>'
             '<sup>Orange = high-risk (≥3% revenue shrinkage) · teal = lower-risk</sup>',
        x=0.5, y=0.95, xanchor='center', yanchor='top',
        font=dict(size=15, color='#28251d')),
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='center', x=0.5),
    plot_bgcolor=BG,
    paper_bgcolor='white',
    height=540, width=1020,
    margin=dict(l=180, r=100, t=90, b=40),
)
fig_a.show()

### Inventory Shrinkage by Department Analysis

The left panel shows the **revenue shrinkage rate** (% of expected revenue
that was never registered); the right panel shows the **absolute revenue
lost** in €. Departments exceeding the 3 % threshold are highlighted in
orange and treated as high-risk zones in the fraud model.

**Key findings:**
- **Dept 2** (Fish & Seafood) has by far the highest revenue shrinkage at
  ~8 %, and the largest absolute loss at ~€46 k, making it the single
  highest-risk department.
- **Dept 14** (Meat & Poultry) follows at ~6.8 % and ~€32 k lost, another
  high-value, high-theft zone consistent with expensive loose-weight items.
- **Dept 12** (Wine & Spirits) and **Dept 4** (Deli & Cheese) both exceed
  4 %, confirming that high unit-value aisles are disproportionately
  affected.
- Low shrinkage departments (< 2 %) such as Dept 1, 3, 5, 6 tend to carry
  lower average item values, offering less incentive for selective scanning.

In [ ]:
# Missing stock item value box plot ──────────────────────────────
dept_order = (missing.groupby('department')['item_value']
              .median().sort_values(ascending=False).index.tolist())

fig_b = go.Figure()
for d in dept_order:
    vals  = missing[missing['department'] == d]['item_value'].values
    color = ORANGE if d in HIGH_RISK_DEPTS else TEAL
    r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
    fig_b.add_trace(go.Box(
        y=vals,
        name=f'{d} {DEPT_ABBR.get(d, "")}',
        marker_color=color,
        line_color=color,
        fillcolor=f'rgba({r},{g},{b},0.25)',
        boxpoints=False,
        hovertemplate=f'Dept {d} {DEPT_MAP.get(d,"")}<br>Value: €%{{y:.2f}}<extra></extra>',
    ))

fig_b.update_layout(
    title=dict(
        text='Missing Stock Item Value Distribution per Department<br>'
             '<sup>Orange = high-risk dept · Box shows IQR · Line = median</sup>',
        x=0.5),
    xaxis=dict(title='Department', tickangle=35, gridcolor='#dcd9d5'),
    yaxis=dict(title='Item Value (€)', gridcolor='#dcd9d5'),
    height=480, width=1000, margin=dict(l=60, r=40, t=90, b=80))
fig_b.show()

### Missing Stock Item Value Distribution per Department

Each box shows the IQR of the individual item values
recorded in `stockage_missing.csv` for that department, sorted by descending
median. This tells us not just *where* items go missing, but *how expensive*
the missing items typically are.

**Key findings:**
- **Dept 14** (Meat & Poultry) has both the widest spread and some of the
  highest single-item values (up to ~€20+), confirming loose-weight
  meat as a prime theft target.
- **Dept 2** (Fish & Seafood) shows a high median with notable outliers,
  consistent with its dominant revenue shrinkage rate.
- **Dept 11** (Organic & Health) and **Dept 12** (Wine & Spirits) display
  elevated medians, reflecting their higher unit prices.
- Departments with narrow, low-value boxes (e.g. Dept 6, Dept 13) lose
  items less costly per event, even if their total item-loss count is
  moderate.

---
## 10. Distance Anomaly Scoring

Using the five distance features already engineered in Section 5, we compute
a **composite distance anomaly score** per customer via Z-score normalisation (using `supermarket.csv`).

**Five distance features used:**

| Feature | What it captures |
|---|---|
| `total_dist` | Total path length walked through the store |
| `avg_dist` | Average distance between consecutive departments |
| `max_dist` | Largest single step — proxy for teleporting / skipping aisles |
| `std_dist` | Variability of step distances — erratic vs systematic shopping |
| `dept_revisits` | Extra distance from revisiting already-visited departments |

A high composite Z-score indicates a customer whose movement is globally
unusual: too short, too long, or too erratic compared to normal shoppers.

### How the Score Is Computed

The `distance_anomaly_score` is a **composite Z-score** derived from five distance features: `total_dist`, `avg_dist`, `max_dist`, `std_dist`, and `dept_revisits`. Each feature is individually Z-scored and the five Z-scores are averaged into a single value per customer. A higher score means the customer's movement pattern is further from the population norm across all five dimensions simultaneously. Customers in the **top 5%** (score > p95 = **1.34**) are flagged as `dist_anomaly_flag = 1`.

In [ ]:
# Distance Anomaly Scoring ──────────────────────────────────────

DIST_FEATURES = [
    'total_dist',
    'avg_dist',
    'max_dist',
    'std_dist',
    'dept_revisits',
]

# Z-score each feature, then average into composite score
df_dist  = df[DIST_FEATURES].copy()
z_scores = df_dist.apply(zscore, nan_policy='omit')
z_scores.columns = [f'z_{c}' for c in DIST_FEATURES]

df[list(z_scores.columns)] = z_scores.values
df['dist_anomaly_score'] = z_scores.mean(axis=1)

# Flag top 5 %
threshold_95 = df['dist_anomaly_score'].quantile(0.95)
df['dist_anomaly_flag'] = (df['dist_anomaly_score'] >= threshold_95).astype(int)

n_flagged = df['dist_anomaly_flag'].sum()
print(f'Threshold (95th pct) : {threshold_95:.4f}')
print(f'Customers flagged    : {n_flagged:,}  ({n_flagged/len(df)*100:.1f}%)')
print(f'\nMean dist anomaly score — flagged : {df[df["dist_anomaly_flag"]==1]["dist_anomaly_score"].mean():.3f}')
print(f'Mean dist anomaly score — normal  : {df[df["dist_anomaly_flag"]==0]["dist_anomaly_score"].mean():.3f}')


### Distribution of Composite Distance Anomaly Score

The histogram shows how the composite Z-score is distributed across all customers. The vertical dashed line marks the 95th-percentile threshold. The long right tail contains the flagged population.

In [ ]:
# Composite distance anomaly score distribution ──────────────────
bins = np.linspace(df['dist_anomaly_score'].min(), df['dist_anomaly_score'].max(), 81)

normal_vals  = df[df['dist_anomaly_flag'] == 0]['dist_anomaly_score']
flagged_vals = df[df['dist_anomaly_flag'] == 1]['dist_anomaly_score']

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=normal_vals,
    xbins=dict(start=bins[0], end=bins[-1], size=bins[1]-bins[0]),
    marker_color=TEAL, opacity=0.85,
    name='Normal customers',
    hovertemplate='Score: %{x:.2f}<br>Count: %{y:,}<extra></extra>',
))

fig.add_trace(go.Histogram(
    x=flagged_vals,
    xbins=dict(start=bins[0], end=bins[-1], size=bins[1]-bins[0]),
    marker_color=ORANGE, opacity=0.85,
    name=f'Flagged — top 5% (≥ {threshold_95:.2f})',
    hovertemplate='Score: %{x:.2f}<br>Count: %{y:,}<extra></extra>',
))

fig.add_vline(
    x=threshold_95,
    line_dash='dash', line_color=RED, line_width=2,
    annotation_text=f'95th pct: {threshold_95:.2f}',
    annotation_position='top right',
    annotation_font=dict(size=11, color=RED),
)

fig.update_layout(
    barmode='overlay',
    title=dict(text='Composite Distance Anomaly Score Distribution',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Composite Z-score (avg of 5 distance features)', gridcolor='#dcd9d5'),
    yaxis=dict(title='Number of Customers', gridcolor='#dcd9d5'),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=760, height=420,
    margin=dict(l=60, r=40, t=60, b=60),
    legend=dict(x=0.65, y=0.95, bgcolor='rgba(0,0,0,0)'),
)
fig.show()

### Composite Distance Anomaly Score Distribution Analysis

**Key observations:**

- The **bulk of customers** (teal) cluster tightly between −1.5 and +1.0,
  reflecting consistent, predictable movement patterns through the store.
- The **flagged population** (orange) occupies the right tail beyond the
  95th-percentile threshold of **1.05**, representing ~5,000 customers whose
  combined distance behaviour is significantly above average.
- The right tail is notably heavier than the left, indicating that anomalous
  shopping paths tend to be *excessively long or erratic* rather than
  unusually short. It is consistent with customers making large, irregular jumps
  between distant departments.
- The clean separation between the two groups from the threshold onwards
  suggests the composite score is a reliable first-stage filter, well-suited
  to feed into the downstream fraud detection model as a binary flag
  (`dist_anomaly_flag = 1`).

### Flagged vs Normal Customers: Feature Ratios

Each bar shows the ratio of the **flagged group mean** to the **normal group mean** for a selection of key features. A ratio > 1 means flagged customers score higher on that feature; < 1 means they score lower. The dashed line at x = 1 is the normal baseline. 

In [ ]:
# Flagged vs Normal — feature ratio chart ────────────────────────
RATIO_FEATURES = [
    'total_dist', 'avg_dist', 'max_dist', 'std_dist',
    'dept_revisits', 'n_large_jumps', 'jump_ratio',
    'n_scans', 'total_amount', 'zero_amounts',
    'fast_scans', 'fast_scan_ratio', 'long_dwells',
    'high_value_scans', 'fast_high_value', 'low_amount_ratio',
]

flagged_means = df[df['dist_anomaly_flag'] == 1][RATIO_FEATURES].mean()
normal_means  = df[df['dist_anomaly_flag'] == 0][RATIO_FEATURES].mean()

# Avoid division by zero
ratio = (flagged_means / normal_means.replace(0, np.nan)).dropna()
ratio = ratio.sort_values(ascending=True) 

def bar_color(v):
    if v > 1.05:   return ORANGE
    elif v < 0.95: return TEAL
    else:          return MUTED

colors = [bar_color(v) for v in ratio.values]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=ratio.index,
    x=ratio.values,
    orientation='h',
    marker_color=colors,
    text=[f'{v:.2f}×' for v in ratio.values],
    textposition='outside',
    textfont=dict(size=10),
    cliponaxis=False,
    hovertemplate='%{y}<br>Ratio: %{x:.3f}×<extra></extra>',
))

fig.add_vline(
    x=1, line_dash='dash', line_color=MUTED, line_width=1.5,
    annotation_text='Baseline (normal = 1)',
    annotation_position='top right',
    annotation_font=dict(size=10, color=MUTED),
)

fig.update_layout(
    title=dict(text='Flagged vs. Normal Customers — Feature Ratios',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Flagged / Normal ratio', gridcolor='#dcd9d5', range=[0, ratio.max() * 1.18]),
    yaxis=dict(tickfont=dict(size=11)),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=760, height=500,
    margin=dict(l=140, r=80, t=60, b=60),
    showlegend=False,
)
fig.show()

In [ ]:
# Mean Z-score per feature — flagged vs normal ───────────────────
z_cols    = [f'z_{c}' for c in DIST_FEATURES]
feat_labels = [c.replace('z_', '').replace('_', ' ').title() for c in z_cols]

flagged_means = df[df['dist_anomaly_flag'] == 1][z_cols].mean().values
normal_means  = df[df['dist_anomaly_flag'] == 0][z_cols].mean().values

fig = go.Figure()
fig.add_trace(go.Bar(
    x=feat_labels, y=flagged_means,
    name='Flagged (top 5%)', marker_color=ORANGE, opacity=0.85,
    hovertemplate='%{x}<br>Mean Z: %{y:.3f}<extra></extra>',
))
fig.add_trace(go.Bar(
    x=feat_labels, y=normal_means,
    name='Normal', marker_color=TEAL, opacity=0.85,
    hovertemplate='%{x}<br>Mean Z: %{y:.3f}<extra></extra>',
))

fig.add_hline(y=0, line_color=MUTED, line_width=1)

fig.update_layout(
    barmode='group',
    title=dict(text='Mean Distance Feature Z-Scores: Flagged vs Normal Customers',
               font=dict(size=15, color='#28251d'), x=0.5),
    xaxis=dict(title='Distance Feature', gridcolor='#dcd9d5'),
    yaxis=dict(title='Mean Z-Score', gridcolor='#dcd9d5', zeroline=False),
    plot_bgcolor=BG, paper_bgcolor='white',
    width=760, height=420,
    margin=dict(l=60, r=40, t=60, b=80),
    legend=dict(x=0.75, y=0.95, bgcolor='rgba(0,0,0,0)'),
)
fig.show()